# KG1 v142 — Nemotron-3-Nano-30B SFT (Colab Pro+ A100/H100)

## Objetivo: TARGET 0.92 (consensus 28 APIs triple check)

**Datasets (~11,402 CoTs total, public, sem auth):**
- Tong corpus → 6,014 (`felipesp1983/nemotron-cot-tong-dgxchen`)
- Cryptarithm 813 → 813 (`felipesp1983/nemotron-cryptarithm-cot-813`)
- EqGuess 150 → 150 (gist DeepSeek-R1 distill)
- Synth Z3 4425 → 4,425 (`felipesp1983/nemotron-cryptarithm-synth-4425`)

**Hyperparams (consenso triple check):**
- LoRA r=32 alpha=32 all-linear+lm_head
- lr=5e-5 (consenso 23 APIs - era 2e-4 no Tong)
- epochs=2 (consenso 18 APIs - era 1 no Tong)
- cosine scheduler, warmup=0.03, weight_decay=0.01
- max_length=8192 (Kaggle limit), batch effective 32
- Chat template enable_thinking=True
- Prompt masking labels=-100
- Curriculum: tong → eq_guess → cryptarithm_813 → synth_4425

## Setup obrigatorio

1. **Runtime → Change runtime type → A100** (HighRAM se aparecer) ou **H100**
2. **Colab Secrets** (icone chave lateral esquerdo):
   - `HF_KEY` (HuggingFace token com Write+Jobs - usado SOMENTE no Cell 9)
   - `KAGGLE_USERNAME`
   - `KAGGLE_KEY`
3. **Runtime → Run all** (Ctrl+F9)

Tempo: **3-6 horas**

## Versoes verificadas (atualizadas Apr 2026)

- mamba-ssm: **2.3.1** (suporte torch 2.8/2.9/2.10/2.11)
- causal-conv1d: **1.6.1** (release tag v1.6.1.post4)
- transformers: 4.48+
- peft: 0.14+, trl: 0.14+, bitsandbytes: 0.45+

## Cells self-contained

Cada cell tem seus proprios imports - pode rodar isoladamente apos restart.


In [ ]:
# CELL 1: Setup secrets + GPU check
import os, sys

try:
    from google.colab import userdata
    IS_COLAB = True
except ImportError:
    IS_COLAB = False
    print('[WARN] Not running in Colab')

# HF token: HF_KEY preferred, HF_TOKEN fallback
hf_token = None
if IS_COLAB:
    for key_name in ['HF_KEY', 'HF_TOKEN']:
        try:
            v = userdata.get(key_name)
            if v:
                hf_token = v
                os.environ['HF_TOKEN'] = v
                print(f'HF token set via {key_name} (len={len(v)})')
                break
        except Exception:
            continue
    if not hf_token:
        print('[WARN] HF_KEY/HF_TOKEN not set in Colab Secrets')
        print('       Configure before Cell 9 (upload adapter)')

    # Kaggle (optional for training, needed for submit later)
    try:
        os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
        os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
        print(f'Kaggle creds: {os.environ["KAGGLE_USERNAME"]}')
    except Exception:
        print('[INFO] Kaggle creds not set (only needed for kaggle CLI)')

# GPU check
import torch
print()
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.version.cuda}')
print(f'cxx11_abi: {torch.compiled_with_cxx11_abi()}')

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {gpu_name}')
    print(f'VRAM: {vram:.1f} GB')
else:
    print('[ERROR] No GPU detected - go to Runtime > Change runtime type > GPU')
    raise RuntimeError('GPU required')

import psutil
print(f'RAM: {psutil.virtual_memory().total/1e9:.1f} GB')


In [ ]:
# CELL 2: Install dependencies (Colab has CUDA + PyTorch preinstalled)
%pip install -q --upgrade transformers>=4.48.0 peft>=0.14.0 trl>=0.14.0 datasets>=3.0.0 accelerate>=1.3.0 bitsandbytes>=0.45.0 huggingface_hub>=0.27.0
%pip install -q einops packaging ninja triton

# Verify
import transformers, peft, trl, datasets, accelerate, bitsandbytes
print(f'transformers:  {transformers.__version__}')
print(f'peft:          {peft.__version__}')
print(f'trl:           {trl.__version__}')
print(f'datasets:      {datasets.__version__}')
print(f'accelerate:    {accelerate.__version__}')
print(f'bitsandbytes:  {bitsandbytes.__version__}')


In [ ]:
# CELL 3: Install mamba-ssm v2.3.1 + causal-conv1d v1.6.1 (Colab atual: cp312 torch2.10)
import sys, subprocess
import torch

# Detect Python + Torch + ABI
py_ver = f'cp{sys.version_info.major}{sys.version_info.minor}'  # cp312
torch_short = '.'.join(torch.__version__.split('+')[0].split('.')[:2])  # 2.10
abi_str = 'TRUE' if torch.compiled_with_cxx11_abi() else 'FALSE'
cuda_str = 'cu12'  # works for any cuda 12.x
print(f'Python: {py_ver} | Torch: {torch_short} | CUDA: {cuda_str} | ABI: {abi_str}')

MAMBA_VER = '2.3.1'
CC_TAG = '1.6.1.post4'  # github release tag
CC_ASSET_VER = '1.6.1'  # filename version (sem post4)

def install_combo(cu, t, abi):
    cc_url = f'https://github.com/Dao-AILab/causal-conv1d/releases/download/v{CC_TAG}/causal_conv1d-{CC_ASSET_VER}+{cu}torch{t}cxx11abi{abi}-{py_ver}-{py_ver}-linux_x86_64.whl'
    mb_url = f'https://github.com/state-spaces/mamba/releases/download/v{MAMBA_VER}/mamba_ssm-{MAMBA_VER}+{cu}torch{t}cxx11abi{abi}-{py_ver}-{py_ver}-linux_x86_64.whl'
    for url in [cc_url, mb_url]:
        r = subprocess.run(
            [sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', '--no-deps', url],
            capture_output=True, text=True, timeout=600
        )
        if r.returncode != 0:
            return False, r.stderr[-200:]
    return True, ''

# Detected combo first, then fallbacks
attempts = [(cuda_str, torch_short, abi_str)]
opposite_abi = 'FALSE' if abi_str == 'TRUE' else 'TRUE'
attempts.append((cuda_str, torch_short, opposite_abi))
for t in ['2.10', '2.9', '2.8']:
    for abi in ['TRUE', 'FALSE']:
        combo = (cuda_str, t, abi)
        if combo not in attempts:
            attempts.append(combo)

success = False
for cu, t, abi in attempts:
    print(f'Trying: {cu} torch{t} abi{abi}...')
    ok, err = install_combo(cu, t, abi)
    if ok:
        print(f'  [OK] Installed: mamba_ssm {MAMBA_VER} + causal_conv1d {CC_ASSET_VER}')
        success = True
        break
    else:
        print(f'  [FAIL] {err[:120].replace(chr(10), " ")}')

if not success:
    print()
    print('All wheel combos failed. Last fallback: build from source...')
    import os
    os.environ['CAUSAL_CONV1D_FORCE_BUILD'] = 'TRUE'
    os.environ['MAMBA_FORCE_BUILD'] = 'TRUE'
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-build-isolation',
                    '--force-reinstall', f'causal-conv1d=={CC_ASSET_VER}'], timeout=1200)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-build-isolation',
                    '--force-reinstall', f'mamba-ssm=={MAMBA_VER}'], timeout=1500)

# Force reload modules
for m in ['mamba_ssm', 'causal_conv1d', 'selective_scan_cuda', 'causal_conv1d_cuda']:
    if m in sys.modules:
        del sys.modules[m]

# Verify imports + CUDA kernels
print()
print('=== Verification ===')
all_ok = True
for m in ['mamba_ssm', 'causal_conv1d', 'selective_scan_cuda', 'causal_conv1d_cuda']:
    try:
        mod = __import__(m)
        v = getattr(mod, '__version__', 'imported')
        print(f'  [OK] {m}: {v}')
    except ImportError as e:
        print(f'  [FAIL] {m}: {e}')
        all_ok = False

if all_ok:
    print()
    print('SUCCESS: All mamba-ssm CUDA kernels available!')
else:
    print()
    print('!!! Some failed - try Runtime > Restart runtime, then re-run Cells 1-3')
    raise RuntimeError('mamba-ssm install incomplete')


In [ ]:
# CELL 4: Download all 4 datasets via direct HTTP (sem precisar de token HF)
import os, urllib.request

DATA_DIR = '/content/data'
os.makedirs(DATA_DIR, exist_ok=True)

URLS = {
    'tong.csv': 'https://huggingface.co/datasets/felipesp1983/nemotron-cot-tong-dgxchen/resolve/main/less_cot.csv',
    'cryptarithm_813.jsonl': 'https://huggingface.co/datasets/felipesp1983/nemotron-cryptarithm-cot-813/resolve/main/cryptarithm_cot_813.jsonl',
    'eq_guess_150.jsonl': 'https://gist.githubusercontent.com/FELIPEACASTRO/0d4674009f3886d6add5be636292001a/raw',
    'synth_4425.jsonl': 'https://huggingface.co/datasets/felipesp1983/nemotron-cryptarithm-synth-4425/resolve/main/synth_cryptarithm_verified.jsonl',
}

for name, url in URLS.items():
    out = os.path.join(DATA_DIR, name)
    print(f'Downloading {name}...')
    urllib.request.urlretrieve(url, out)
    sz = os.path.getsize(out)
    if sz > 1e6:
        print(f'  [OK] {name}: {sz/1e6:.2f} MB')
    else:
        print(f'  [OK] {name}: {sz/1e3:.1f} KB')

# Globals para outros cells
tong_path = os.path.join(DATA_DIR, 'tong.csv')
crypt_path = os.path.join(DATA_DIR, 'cryptarithm_813.jsonl')
eq_guess_path = os.path.join(DATA_DIR, 'eq_guess_150.jsonl')
synth_path = os.path.join(DATA_DIR, 'synth_4425.jsonl')

print()
print('All paths set. Continue to Cell 5.')


In [ ]:
# CELL 5: Format + Merge + Curriculum sort
import os, csv, json
from datasets import Dataset

# Garantir que paths existem (caso Cell 4 nao tenha sido rodado nesta sessao)
DATA_DIR = '/content/data'
tong_path = os.path.join(DATA_DIR, 'tong.csv')
crypt_path = os.path.join(DATA_DIR, 'cryptarithm_813.jsonl')
eq_guess_path = os.path.join(DATA_DIR, 'eq_guess_150.jsonl')
synth_path = os.path.join(DATA_DIR, 'synth_4425.jsonl')
for p in [tong_path, crypt_path, eq_guess_path, synth_path]:
    if not os.path.exists(p):
        raise FileNotFoundError(f'Run Cell 4 first - {p} not found')

PROMPT_SUFFIX = '\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`'

all_data = []

# 1. Tong (6014)
n_tong = 0
with open(tong_path, encoding='utf-8') as f:
    rd = csv.DictReader(f)
    for row in rd:
        prompt = (row.get('prompt') or '').strip()
        cot = (row.get('generated_cot') or '').strip()
        if prompt and cot:
            all_data.append({
                'prompt': prompt + PROMPT_SUFFIX,
                'completion': cot,
                'source': 'tong',
            })
            n_tong += 1

# 2. Cryptarithm 813
n_crypt = 0
with open(crypt_path, encoding='utf-8') as f:
    for line in f:
        row = json.loads(line)
        if row.get('is_valid') and row.get('cot'):
            all_data.append({
                'prompt': row['prompt'].strip() + PROMPT_SUFFIX,
                'completion': row['cot'].strip(),
                'source': 'cryptarithm_813',
            })
            n_crypt += 1

# 3. EqGuess 150
n_eq = 0
with open(eq_guess_path, encoding='utf-8') as f:
    for line in f:
        row = json.loads(line)
        prompt = (row.get('prompt') or '').strip()
        cot = (row.get('generated_cot') or '').strip()
        if prompt and cot:
            all_data.append({
                'prompt': prompt + PROMPT_SUFFIX,
                'completion': cot,
                'source': 'eq_guess_150',
            })
            n_eq += 1

# 4. Synth Z3 4425
n_synth = 0
with open(synth_path, encoding='utf-8') as f:
    for line in f:
        row = json.loads(line)
        prompt = (row.get('prompt') or '').strip()
        cot = (row.get('generated_cot') or '').strip()
        if prompt and cot:
            all_data.append({
                'prompt': prompt + PROMPT_SUFFIX,
                'completion': cot,
                'source': 'synth_4425',
            })
            n_synth += 1

print(f'Tong:               {n_tong}')
print(f'Cryptarithm 813:    {n_crypt}')
print(f'EqGuess 150:        {n_eq}')
print(f'Synth Z3 4425:      {n_synth}')
print(f'TOTAL:              {len(all_data)}')

# Curriculum: easy first (Tong) -> hard (cryptarithm + synth)
SOURCE_ORDER = {'tong': 0, 'eq_guess_150': 1, 'cryptarithm_813': 2, 'synth_4425': 3}
all_data.sort(key=lambda x: SOURCE_ORDER.get(x['source'], 99))

ds = Dataset.from_list(all_data)
print()
print('Curriculum: tong -> eq_guess -> cryptarithm_813 -> synth_4425')
print(f'Dataset ready: {len(ds)} samples')


In [ ]:
# CELL 6: Load Nemotron-3-Nano-30B + LoRA (auto NF4 if VRAM < 70GB)
import os, time
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# === RELOAD HF token from Colab Secrets (in case it was renewed after Cell 1) ===
try:
    from google.colab import userdata
    for key_name in ['HF_KEY', 'HF_TOKEN']:
        try:
            v = userdata.get(key_name)
            if v:
                os.environ['HF_TOKEN'] = v
                print(f'HF token (re)loaded via {key_name} (len={len(v)})')
                break
        except Exception:
            continue
except ImportError:
    pass

# === VERIFY TOKEN VALID before downloading 60GB model ===
hf_token = os.environ.get('HF_TOKEN')
if hf_token:
    try:
        from huggingface_hub import HfApi
        whoami = HfApi(token=hf_token).whoami()
        print(f'  Token OK, user: {whoami["name"]}')
    except Exception as e:
        print(f'  [ERR] Token invalid/expired: {e}')
        print('  ACAO NECESSARIA:')
        print('  1. Gere novo token em https://huggingface.co/settings/tokens')
        print('  2. Permissoes: Read access to public gated repos + Write access to repos')
        print('  3. Aceite termos em https://huggingface.co/nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16')
        print('  4. Atualize Colab Secret HF_KEY')
        print('  5. Re-rode este Cell')
        raise RuntimeError('HF token invalido - veja instrucoes acima')
else:
    print('[ERR] HF_TOKEN nao configurado - configure Colab Secret HF_KEY')
    raise RuntimeError('HF_KEY/HF_TOKEN missing')

MODEL_NAME = 'nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16'

# Auto-detect VRAM
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
USE_NF4 = vram_gb < 70  # H100 80GB / A100 80GB = no NF4 needed
print(f'\nVRAM: {vram_gb:.1f} GB  ->  USE_NF4: {USE_NF4}')

print('\nLoading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True, token=hf_token)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f'Loading model (NF4={USE_NF4})...')
t0 = time.time()
if USE_NF4:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map={'': 0},
        trust_remote_code=True,
        token=hf_token,
        attn_implementation='eager',
        torch_dtype=torch.bfloat16,
    )
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
else:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.bfloat16,
        device_map='auto',
        trust_remote_code=True,
        token=hf_token,
        attn_implementation='eager',
    )
    model.enable_input_require_grads()
    model.gradient_checkpointing_enable()

print(f'  [OK] Model loaded in {time.time()-t0:.1f}s')
print(f'  VRAM after load: {torch.cuda.memory_allocated()/1e9:.1f} GB')

# Apply LoRA
TARGET_REGEX = r'.*\.(in_proj|out_proj|q_proj|k_proj|v_proj|o_proj|up_proj|down_proj|gate_proj|lm_head)$'
model = get_peft_model(model, LoraConfig(
    r=32,
    lora_alpha=32,
    target_modules=TARGET_REGEX,
    lora_dropout=0.0,
    bias='none',
    task_type='CAUSAL_LM',
))

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'  [OK] LoRA trainable: {trainable/1e6:.1f}M / {total/1e9:.2f}B ({100*trainable/total:.3f}%)')
print(f'  VRAM after LoRA: {torch.cuda.memory_allocated()/1e9:.1f} GB / {vram_gb:.1f} GB total')


In [ ]:
# CELL 7: Tokenize com chat template + prompt masking
# Requires: tokenizer (Cell 6), ds (Cell 5)
MAX_LENGTH = 8192

def fmt(ex):
    messages = [
        {'role': 'user', 'content': ex['prompt']},
        {'role': 'assistant', 'content': ex['completion']},
    ]
    # Prompt-only ids (para mask)
    prompt_ids = tokenizer.apply_chat_template(
        [messages[0]],
        tokenize=True,
        add_generation_prompt=True,
        enable_thinking=True,
    )
    # Full conversation ids
    full_ids = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=False,
        enable_thinking=True,
    )
    if len(full_ids) > MAX_LENGTH:
        full_ids = full_ids[:MAX_LENGTH]

    # Labels: -100 para prompt tokens, real ids para completion
    labels = list(full_ids)
    n_prompt = min(len(prompt_ids), len(labels))
    for i in range(n_prompt):
        labels[i] = -100

    return {
        'input_ids': full_ids,
        'attention_mask': [1] * len(full_ids),
        'labels': labels,
    }

print(f'Tokenizing {len(ds)} samples...')
tokenized = ds.map(fmt, remove_columns=ds.column_names, num_proc=4, desc='Tokenizing')
print(f'[OK] Tokenized: {len(tokenized)} samples')

# Sanity check first sample
sample = tokenized[0]
print(f'\nFirst sample:')
print(f'  input_ids length: {len(sample["input_ids"])}')
print(f'  prompt tokens (masked): {sum(1 for x in sample["labels"] if x == -100)}')
print(f'  completion tokens (trained): {sum(1 for x in sample["labels"] if x != -100)}')


In [ ]:
# CELL 8: TRAIN (lr=5e-5, epochs=2 - consenso triple check 28 APIs)
import os, time
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq

OUTPUT_DIR = '/content/output_v142'
os.makedirs(OUTPUT_DIR, exist_ok=True)

train_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=2,                # consenso 18 APIs (era 1 no Tong)
    per_device_train_batch_size=1,
    gradient_accumulation_steps=32,    # batch effective = 32
    learning_rate=5e-5,                # consenso 23 APIs (era 2e-4 no Tong)
    lr_scheduler_type='cosine',
    warmup_ratio=0.03,
    adam_beta1=0.9,
    adam_beta2=0.95,
    adam_epsilon=1e-8,
    weight_decay=0.01,
    max_grad_norm=1.0,
    logging_steps=10,
    save_steps=200,
    save_total_limit=3,
    bf16=True,
    optim='adamw_torch',
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    remove_unused_columns=False,
    report_to='none',
    dataloader_num_workers=2,
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    padding=True,
    label_pad_token_id=-100,
    pad_to_multiple_of=8,
    return_tensors='pt',
)

trainer = Trainer(
    model=model,
    args=train_args,
    train_dataset=tokenized,
    data_collator=data_collator,
)

print(f'Training {len(tokenized)} samples for 2 epochs (effective batch 32)...')
print(f'Expected: ~{2 * len(tokenized) // 32} optimizer steps')
print()

t0 = time.time()
trainer.train()
train_time_min = (time.time() - t0) / 60
print()
print(f'[OK] Training complete in {train_time_min:.1f} min')


In [ ]:
# CELL 9: Save adapter + Upload to HuggingFace
# REQUER token HF VALIDO em os.environ['HF_TOKEN']
import os
from huggingface_hub import HfApi, create_repo

OUTPUT_DIR = '/content/output_v142'
ADAPTER_DIR = f'{OUTPUT_DIR}/final_adapter'

trainer.save_model(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

print('Saved adapter files:')
for f in os.listdir(ADAPTER_DIR):
    sz = os.path.getsize(os.path.join(ADAPTER_DIR, f))
    print(f'  {f}: {sz/1e6:.2f} MB')

DEST_REPO = 'felipesp1983/kg1-v142-final'
hf_token = os.environ.get('HF_TOKEN')

if not hf_token:
    print()
    print('!!! HF_TOKEN nao configurado nos Colab Secrets')
    print('!!! Adapter salvo localmente apenas. Roda Cell 10 para backup GDrive.')
    print('!!! Depois gere novo token em https://huggingface.co/settings/tokens')
else:
    try:
        api = HfApi(token=hf_token)
        whoami = api.whoami()
        print(f'\nHF user: {whoami["name"]}')
        create_repo(DEST_REPO, private=False, exist_ok=True, token=hf_token)
        api.upload_folder(
            folder_path=ADAPTER_DIR,
            repo_id=DEST_REPO,
            repo_type='model',
            commit_message='v142 SFT Tong+813+150+4425synth lr5e-5 ep2 cosine',
        )
        print(f'\n[OK] Uploaded: https://huggingface.co/{DEST_REPO}')
    except Exception as e:
        print(f'\n[ERR] Upload failed: {e}')
        print('\nGere novo token HF e atualize Colab Secret HF_KEY, depois re-run THIS cell.')
        print('https://huggingface.co/settings/tokens')


In [ ]:
# CELL 10: GDrive backup (sempre roda, mesmo se Cell 9 falhar)
import os, shutil, subprocess
from google.colab import drive

drive.mount('/content/drive')

ADAPTER_DIR = '/content/output_v142/final_adapter'
GDRIVE_BACKUP = '/content/drive/My Drive/KG1_v142_adapter'

if os.path.exists(GDRIVE_BACKUP):
    shutil.rmtree(GDRIVE_BACKUP)
shutil.copytree(ADAPTER_DIR, GDRIVE_BACKUP)

print(f'[OK] Backup: {GDRIVE_BACKUP}')

# Show size + files
sz = subprocess.run(['du', '-sh', GDRIVE_BACKUP], capture_output=True, text=True).stdout
print(f'Total size: {sz.strip()}')
print()
print('Files:')
for f in sorted(os.listdir(GDRIVE_BACKUP)):
    p = os.path.join(GDRIVE_BACKUP, f)
    if os.path.isfile(p):
        sz_f = os.path.getsize(p)
        print(f'  {f}: {sz_f/1e6:.2f} MB')

print()
print('IMPORTANT: keep this Drive backup - se HF upload falhar pode usar essa copia.')


## Proximos passos apos v142 treinar

### 1. Quick eval no Colab (opcional, ~5 min)

```python
# Test adapter generates correct \boxed{} format
test_prompt = "In Alice's Wonderland, a secret set of transformation rules is applied to equations. Below are a few examples:\n%|*\"|  = %|\"|\n(%+[@ = (%[@\nNow, determine the result for: \\(*[#"
inputs = tokenizer(test_prompt + tokenizer.eos_token, return_tensors='pt').to(model.device)
output = model.generate(**inputs, max_new_tokens=512, temperature=0)
print(tokenizer.decode(output[0], skip_special_tokens=False))
```

### 2. Submit Kaggle (terminal local depois)

```bash
python scripts/submit_kaggle.py \\
    --hf-repo felipesp1983/kg1-v142-final \\
    --reference-adapter-dir runs/attached_085_audit_20260416 \\
    --test-csv data/kaggle/unzipped/test.csv \\
    --message "v142 11400CoTs Tong+synth4425 lr5e-5 ep2" \\
    --max-daily-submits 5
```

### 3. Score esperado (consenso 28 APIs)
- **Mediana**: 0.92
- Range: [0.88 - 0.95]
- P(>= 0.87): 75-85%

### 4. Se v142 < 0.87
Proximo: **v143 GRPO** sobre v142 SFT
- Adaptar `rewards.py` do andy279 (open-r1 fork)
- Verifiable reward via Z3 (cryptarithm) + sympy (eq_numeric)
- +5-10% adicional esperado

### Troubleshooting

**Cell 3 falhar**: Mamba/causal-conv1d wheel mismatch. Cells auto-detectam Python+Torch+ABI e tentam 8 combos. Se ainda falhar, build from source rodara (~10 min).

**Cell 6 OOM**: Aumenta `gradient_accumulation_steps` no Cell 8 ou usa NF4 mais agressivo (ja auto-aplicado se VRAM < 70GB).

**Cell 9 token expirado**: Felipe gera novo token em huggingface.co/settings/tokens, atualiza Colab Secret HF_KEY, re-roda Cell 9 (Cell 10 GDrive sempre funciona).

**Loss explode**: Se loss > 5.0 nos primeiros 50 steps, reduz lr para 3e-5 e restart Cell 8.
